<a href="https://colab.research.google.com/github/MohammadRushaan/Computer-Vision-OpenCV/blob/main/12_Face_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import os
import cv2
import numpy as np
from urllib.request import urlretrieve
from zipfile import ZipFile
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import PIL.Image

# 1. Download Model Assets
URL = r"https://www.dropbox.com/s/efitgt363ada95a/opencv_bootcamp_assets_12.zip?dl=1"
asset_zip_path = os.path.join(os.getcwd(), "opencv_bootcamp_assets_12.zip")

if not os.path.exists("deploy.prototxt") or not os.path.exists("res10_300x300_ssd_iter_140000_fp16.caffemodel"):
    print("Downloading assets...")
    urlretrieve(URL, asset_zip_path)
    with ZipFile(asset_zip_path) as z:
        z.extractall(os.getcwd())
    print("Assets downloaded and extracted.")

# 2. Load the Caffe Face Detection Model
net = cv2.dnn.readNetFromCaffe(
    "deploy.prototxt",
    "res10_300x300_ssd_iter_140000_fp16.caffemodel"
)

# 3. Setup Browser Streaming Interface
def init_camera():
    js = Javascript('''
        async function setupCamera() {
            const container = document.createElement('div');
            container.id = 'webcam-container';
            container.style.position = 'relative';
            container.style.display = 'inline-block';

            const video = document.createElement('video');
            video.id = 'webcam-video';
            video.autoplay = true;
            video.muted = true;
            video.playsInline = true;
            video.style.display = 'block';

            const canvas = document.createElement('canvas');
            canvas.id = 'overlay-canvas';
            canvas.style.position = 'absolute';
            canvas.style.top = '0';
            canvas.style.left = '0';
            canvas.style.pointerEvents = 'none';

            const stopBtn = document.createElement('button');
            stopBtn.id = 'stop-btn';
            stopBtn.textContent = 'Stop Stream';
            stopBtn.style.display = 'block';
            stopBtn.style.margin = '10px 0';
            stopBtn.style.padding = '8px 16px';
            stopBtn.style.backgroundColor = '#d9534f';
            stopBtn.style.color = 'white';
            stopBtn.style.border = 'none';
            stopBtn.style.borderRadius = '4px';
            stopBtn.style.cursor = 'pointer';

            container.appendChild(video);
            container.appendChild(canvas);
            document.body.appendChild(stopBtn);
            document.body.appendChild(container);

            const stream = await navigator.mediaDevices.getUserMedia({
                video: { width: 640, height: 480 }
            });
            video.srcObject = stream;
            await video.play();

            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            window.webcamStream = stream;
            window.isStreamingStopped = false;

            stopBtn.onclick = () => {
                window.isStreamingStopped = true;
                window.webcamStream.getVideoTracks().forEach(track => track.stop());
                container.remove();
                stopBtn.remove();
            };

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        }
        setupCamera();
    ''')
    display(js)

def grab_frame():
    js = Javascript('''
        async function getFrame() {
            if (window.isStreamingStopped || !window.webcamStream) {
                return null;
            }
            const video = document.getElementById('webcam-video');
            if (!video) return null;

            const tempCanvas = document.createElement('canvas');
            tempCanvas.width = video.videoWidth;
            tempCanvas.height = video.videoHeight;
            const ctx = tempCanvas.getContext('2d');
            ctx.drawImage(video, 0, 0);
            return tempCanvas.toDataURL('image/jpeg', 0.6);
        }
        getFrame();
    ''')
    display(js)
    return eval_js('getFrame()')

def render_detections(overlay_b64):
    js = Javascript(f'''
        async function drawOverlay() {{
            const canvas = document.getElementById('overlay-canvas');
            if (!canvas) return;
            const ctx = canvas.getContext('2d');
            const img = new Image();
            img.onload = () => {{
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                ctx.drawImage(img, 0, 0);
            }};
            img.src = "{overlay_b64}";
        }}
        drawOverlay();
    ''')
    display(js)

# 4. Start Streaming Loop
init_camera()

import time
time.sleep(1.5)  # Allow browser camera to initialize

while True:
    frame_data = grab_frame()
    if frame_data is None:
        print("Stream ended.")
        break

    # Convert Base64 frame to OpenCV BGR
    img_bytes = b64decode(frame_data.split(',')[1])
    np_arr = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

    if frame is None:
        continue

    h, w = frame.shape[:2]
    overlay = np.zeros((h, w, 4), dtype=np.uint8)

    # Preprocess & Run SSD Face Detector
    blob = cv2.dnn.blobFromImage(
        frame, 1.0, (300, 300), [104.0, 177.0, 123.0], swapRB=False, crop=False
    )
    net.setInput(blob)
    detections = net.forward()

    # Parse Detections
    for i in range(detections.shape[2]):
        confidence = float(detections[0, 0, i, 2])
        if confidence > 0.5:  # Lowered threshold for reliable capture
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")

            # Clamp coordinates to canvas boundaries
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w - 1, x2), min(h - 1, y2)

            # Draw green bounding box (RGBA)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0, 255), 2)
            label = f"Face: {confidence * 100:.1f}%"
            cv2.putText(
                overlay, label, (x1, max(y1 - 10, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0, 255), 2
            )

    # Convert overlay to Base64 PNG and render
    pil_img = PIL.Image.fromarray(overlay, 'RGBA')
    buff = io.BytesIO()
    pil_img.save(buff, format='PNG')
    overlay_b64 = 'data:image/png;base64,' + b64encode(buff.getvalue()).decode('utf-8')

    render_detections(overlay_b64)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/tmp/ipykernel_2996/260956518.py:182: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_img = PIL.Image.fromarray(overlay, 'RGBA')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Stream ended.
